# ChatInterface + 대화 횟수 세기
- `gr.ChatInterface`로 대화형 채팅 화면
- `history` 매개변수를 실제로 활용해서 "지금까지 몇 번 대화했는지" 세는 기능을 추가

In [1]:
import gradio as gr

# ChatInterface에 넘기는 함수는 (message, history) 두 개를 받음
# message : 사용자가 방금 입력한 메세지
# history : 지금까지의 대화 기록 (현재 메세지는 포함 안 됨)
def echo_bot(message,history):
    turn_count = len(history)// 2+1
    # history는 [{"role": "user", "content":...}, {"role": "assistant", "content":...}]
    # 1회 턴이 user+assistant 2개의 항복을 반환 -> //2 를 하는 이유

    return f"[{turn_count}번째 대화] 너가 말한건 '{message}'이지?"

/Users/soosungkim/workspace/pythonsource/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
demo = gr.ChatInterface(
    fn=echo_bot,
    title="대화 횟수를 세는 에코 챗봇",
    description="입력한 말을 그대로 따라 하면서, 지금이 몇 번째 대환인지도 함께 올려줍니다."
)
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [3]:
demo.close()

Closing server running on port: 7860


In [3]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from google import genai
import gradio as gr

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY")) # OpenAI api



def chat_with_get(message, history):
    message = [{"role": h["role"], "content": h["content"]} for h in history]
    message.append({"role":"user", "content":message})

    response = client.chat.completions.create(
        model = "gpt-4o-mini",
        messages=message
    )
 
    return response.choices[0].message.content

demo = gr.ChatInterface(fn=chat_with_get, title="ChatGPT와 대화하기")
demo.launch()

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [4]:
from transformers import pipeline
import gradio as gr

chatbot = pipeline("text-generation", model="microsoft/DialoGPT-medium")

def chat_local(message,history):
    result = chatbot(message, max_length=100)
    return result[0]["generated_text"]
demo = gr.ChatInterface(fn=chat_local, title="로컬 모델 챗봇")
demo.launch()

Loading weights: 100%|██████████| 293/293 [00:00<00:00, 33066.89it/s]


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


[transformers] Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=100) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=256) and `max_length`

In [5]:
demo.close()

Closing server running on port: 7861
